# Task 3 — Sentimen, Negasi, & Sarkasme (Google Colab Runner)

Notebook ini menjalankan seluruh eksperimen **T3** sesuai rancangan:
- **E1 · Baseline:** Lexicon sentimen + aturan negasi & sarkasme
- **E2 · Transformer:** Fine-tuning **IndoBERTweet** (`indolem/indobertweet-base-uncased`) multi-head (3 seed)
- **E3 · Model Utama:** IndoBERTweet + Negation & Sarcasm Rule Engine
- **E4 · Ablation:** E3 tanpa aturan negasi & tanpa aturan sarkasme

In [ ]:
!nvidia-smi

In [ ]:
import os

REPO_URL = "https://github.com/<USERNAME>/<REPO_NAME>.git"
PROJECT_DIR = "/content/Tubes_NLP"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git status -s

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path

master_path = Path("data/processed/master_dataset.jsonl")
if not master_path.exists():
    print("master_dataset.jsonl belum ada, mengunduh & membangun dataset...")
    !python scripts/download_datasets.py
    !python scripts/generate_synthetic_edge_cases.py
    !python scripts/build_master_dataset.py
else:
    print(f"Dataset siap: {master_path} ({master_path.stat().st_size / 1e6:.2f} MB)")

In [ ]:
!python -m src.task3.train_baseline

In [ ]:
!python -m src.task3.train_indobertweet \
    --model-name indolem/indobertweet-base-uncased \
    --seeds 42 123 2025 \
    --epochs 4 \
    --batch-size 32 \
    --lr 2e-5 \
    --max-length 128 \
    --fp16

In [ ]:
!python -m src.task3.evaluate_t3 --split test

In [ ]:
from src.task3.predictor import predict

samples = [
    "Mantap, lampu jalan depan sekolah mati terus. Semalam ada anak hampir tertabrak.",
    "Bukan tidak menghargai kerja Dishub, tapi Jl. Melati belum pernah diperbaiki sampai sekarang.",
    "Terima kasih banyak untuk Puskesmas Tembalang, pelayanan hari ini cepat dan ramah.",
]
for text in samples:
    print(text)
    print(" ->", predict(text))

In [ ]:
!zip -rq task3_colab_results.zip outputs/ models/task3/
print("Siap diunduh: task3_colab_results.zip")